# Analyze EIBP Log Results

## Input Required Information

| Variable | Use |
| --- | --- |
| LOG_DIR_PATH | Location of the log directory. |

In [16]:
LOG_DIR_PATH = "/home/fabric/work/EIBP/Scripts/local_books/Small_Topology/logs_large_13node"
END_TIME = 1742230487.063541

Convergence time: time taken from 1 failure node to the last node heard about it.

Control Overheard: Size of the failure message overall

Churn rate : how far the error message propagates.

IF_DIRECT_DOWN: nodes detect interface is down

IF_TIMER_DOWN:dead timer is down or expiring.

# Convergence Time

In [17]:
import re
import os

def extract_if_direct_down_time(log_dir_path):
    """Extracts the first 'IF_DIRECT_DOWN' timestamp from the logs."""
    all_files = sorted(os.listdir(log_dir_path))
    for filename in all_files:
        file_path = os.path.join(log_dir_path, filename)
        if os.path.isfile(file_path):
            try:
                with open(file_path, 'r') as file:
                    for line in file:
                        if "IF_DIRECT_DOWN" in line:
                            match = re.search(r'IF_DIRECT_DOWN:(\d+\.\d+)', line)
                            if match:
                                return float(match.group(1)), file_path
            except Exception as e:
                print(f"Error reading file {file_path}: {e}")
    return None, None  # If no match is found

def extract_delete_messages(log_dir_path, start_time, end_time):
    """Extracts all DELETE messages from all files, filtering by time range."""
    delete_messages_per_file = {}
    all_delete_messages = []
    all_files = sorted(os.listdir(log_dir_path))

    for filename in all_files:
        file_path = os.path.join(log_dir_path, filename)
        if os.path.isfile(file_path):
            delete_messages = []
            try:
                with open(file_path, 'r') as file:
                    for line in file:
                        if ("MESSAGE_TYPE_MY_LABELS_DELETE" in line or
                            "MESSAGE_TYPE_PUBLISH_IP_DELETE" in line or
                            "Received MESSAGE_TYPE_PUBLISH_IP_DELETE" in line):

                            match = re.search(r'CURRENT_TIME:(\d+\.\d+)', line)
                            if match:
                                current_time = float(match.group(1))
                                if start_time <= current_time < end_time:  # Ensure it's within the range
                                    delete_messages.append((current_time, line.strip()))
                                    all_delete_messages.append((current_time, file_path, line.strip()))
            except Exception as e:
                print(f"Error reading file {file_path}: {e}")

            if delete_messages:
                delete_messages.sort()  # Sort within each file
                delete_messages_per_file[file_path] = delete_messages

    all_delete_messages.sort()  # Sort globally across all files
    return delete_messages_per_file, all_delete_messages

def calculate_convergence(log_dir_path, end_time):
    """Finds convergence time between IF_DIRECT_DOWN and the latest DELETE message before END_TIME."""
    if_down_time, if_down_file = extract_if_direct_down_time(log_dir_path)

    if if_down_time is None:
        print("Failed to extract 'IF_DIRECT_DOWN' timestamp.")
        return

    print(f"Extracted 'IF_DIRECT_DOWN' timestamp: {if_down_time} in file: {if_down_file}")

    delete_messages_per_file, all_delete_messages = extract_delete_messages(log_dir_path, if_down_time, end_time)

    if not all_delete_messages:
        print("\nNo valid DELETE messages found in the given time range.")
        return

    # Print DELETE messages per file
    print("\n===== DELETE Messages Per File =====")
    for file, messages in delete_messages_per_file.items():
        print(f"\nFile: {file}")
        for timestamp, message in messages:
            print(f"  {timestamp:.2f} - {message}")

    # Find the latest DELETE message across all files
    latest_delete_time, latest_delete_file, latest_delete_message = all_delete_messages[-1]

    print(f"\nLatest DELETE Message before END_TIME: {latest_delete_time:.2f} in file: {latest_delete_file}")

    # Compute convergence time
    convergence_time = latest_delete_time - if_down_time
    print(f"\nConvergence Time: {convergence_time:.2f} seconds ({convergence_time * 1000:.2f} ms)")

# Example usage
calculate_convergence(LOG_DIR_PATH, END_TIME)


Extracted 'IF_DIRECT_DOWN' timestamp: 1742230443.816189 in file: /home/fabric/work/EIBP/Scripts/local_books/Small_Topology/logs_large_13node/EIBP_C1.log

===== DELETE Messages Per File =====

File: /home/fabric/work/EIBP/Scripts/local_books/Small_Topology/logs_large_13node/EIBP_A1.log
  1742230443.86 - Received MESSAGE_TYPE_PUBLISH_IP_DELETE on : eth2 at CURRENT_TIME:1742230443.858792
  1742230443.87 - Received MESSAGE_TYPE_PUBLISH_IP_DELETE on : eth1 at CURRENT_TIME:1742230443.871067

File: /home/fabric/work/EIBP/Scripts/local_books/Small_Topology/logs_large_13node/EIBP_A2.log
  1742230443.83 - Received MESSAGE_TYPE_PUBLISH_IP_DELETE on : eth2 at CURRENT_TIME:1742230443.833090
  1742230443.87 - Received MESSAGE_TYPE_PUBLISH_IP_DELETE on : eth1 at CURRENT_TIME:1742230443.865038

File: /home/fabric/work/EIBP/Scripts/local_books/Small_Topology/logs_large_13node/EIBP_A3.log
  1742230443.84 - Received MESSAGE_TYPE_PUBLISH_IP_DELETE on : eth1 at CURRENT_TIME:1742230443.843870
  1742230443.8

# Control Overhead

In [19]:
import re
import os

# Regular expression to extract CURRENT_TIME from any part of the line
pattern_CURRENT_TIME = re.compile(r'^CURRENT_TIME: (\d+\.\d+)$')

# Initialize a variable to store the overall total ETH_SIZE sum for all files
total_eth_sizes = 0

# Read all log files in the specified directory
for filename in os.listdir(LOG_DIR_PATH):
    if filename.endswith(".log"):  # Process only .log files
        file_path = os.path.join(LOG_DIR_PATH, filename)
        eth_sizes = []  # List to store ETH_SIZE values for this file
        capture_eth_size = False  # Flag to capture ETH_SIZE values
        stop_processing = False  # Flag to stop processing after CURRENT_TIME exceeds END_TIME

        # Read the log file
        with open(file_path, 'r') as file:
            current_time = None  # Initialize the current time
            line_number = 0  # Initialize line number

            # Parse the log to extract ETH_SIZE values
            for line in file:
                line_number += 1  # Increment line number with each line read

                # Check if the line contains CURRENT_TIME
                match_CURRENT_TIME = pattern_CURRENT_TIME.search(line.strip())
                if match_CURRENT_TIME:
                    # Update current time for this log line
                    current_time = float(match_CURRENT_TIME.group(1))
                    # If current time exceeds END_TIME, stop processing further
                    if current_time >= END_TIME:
                        stop_processing = True  # Set the flag to stop processing
                        print(f"Reached END_TIME at line {line_number} in file '{filename}', stopping further ETH_SIZE calculations.")
                        break  # Exit the loop if current_time exceeds END_TIME

                # If stop_processing is set, do not process ETH_SIZE values
                if stop_processing:
                    break  # Stop processing any further lines

                # Capture ETH_SIZE if the log line indicates a delete message
                if "Received MESSAGE_TYPE_PUBLISH_IP_DELETE" in line or "Received MESSAGE_TYPE_MY_LABELS_DELETE" in line:
                    capture_eth_size = True  # Set flag to capture ETH_SIZE values after this line
                elif capture_eth_size and "ETH_SIZE" in line:
                    try:
                        eth_size = line.split(":")[1].strip()
                        eth_sizes.append(int(eth_size))
                        print(f"\nLine {line_number} in file '{filename}': ETH_SIZE = {eth_size} bytes")  # Print ETH_SIZE with line number
                    except (IndexError, ValueError):
                        print(f"Error parsing ETH_SIZE in line {line_number} of file '{filename}': {line}")
                elif capture_eth_size and ("Received MESSAGE_TYPE_PUBLISH_IP_DELETE" in line or "Received MESSAGE_TYPE_MY_LABELS_DELETE" in line):
                    capture_eth_size = False  # Reset the flag if another DELETE MESSAGE line is encountered

        # Calculate the total ETH_SIZE for the current file
        sum_eth_sizes = sum(eth_sizes)
        total_eth_sizes += sum_eth_sizes  # Add the current file's ETH_SIZE to the overall total
        
        # Print the total ETH_SIZE for the selected file
        print(f"\nTotal ETH_SIZE for file '{filename}': {sum_eth_sizes} bytes")

# Print the overall total ETH_SIZE across all files
print(f"\nOverall total ETH_SIZE for all files: {total_eth_sizes} bytes")



Line 2005 in file 'EIBP_C3.log': ETH_SIZE = 22 bytes

Line 2108 in file 'EIBP_C3.log': ETH_SIZE = 32 bytes

Total ETH_SIZE for file 'EIBP_C3.log': 54 bytes

Total ETH_SIZE for file 'EIBP_C1.log': 0 bytes

Line 396 in file 'EIBP_A4.log': ETH_SIZE = 32 bytes

Line 408 in file 'EIBP_A4.log': ETH_SIZE = 32 bytes

Line 458 in file 'EIBP_A4.log': ETH_SIZE = 22 bytes

Total ETH_SIZE for file 'EIBP_A4.log': 86 bytes

Line 422 in file 'EIBP_A1.log': ETH_SIZE = 38 bytes

Line 437 in file 'EIBP_A1.log': ETH_SIZE = 38 bytes

Total ETH_SIZE for file 'EIBP_A1.log': 76 bytes

Line 948 in file 'EIBP_D3.log': ETH_SIZE = 32 bytes

Total ETH_SIZE for file 'EIBP_D3.log': 32 bytes

Line 419 in file 'EIBP_A2.log': ETH_SIZE = 32 bytes

Line 469 in file 'EIBP_A2.log': ETH_SIZE = 38 bytes

Total ETH_SIZE for file 'EIBP_A2.log': 70 bytes

Line 892 in file 'EIBP_D5.log': ETH_SIZE = 32 bytes

Line 935 in file 'EIBP_D5.log': ETH_SIZE = 38 bytes

Total ETH_SIZE for file 'EIBP_D5.log': 70 bytes

Line 1001 in file '

# CHURN PERCENTAGE

In [20]:
import os
import re

# Regular expression to extract CURRENT_TIME from log lines
# pattern_CURRENT_TIME = re.compile(r'^CURRENT_TIME: (\d+\.\d+)$')
pattern_CURRENT_TIME = re.compile(r'CURRENT_TIME:\s*(\d+\.\d+)')
churn_count = 0
total_files = 0

try:
    # Loop through all files in the specified directory
    for filename in os.listdir(LOG_DIR_PATH):
        if filename.endswith('.log'):
            total_files += 1
            # Open the file and check for delete or update messages
            with open(os.path.join(LOG_DIR_PATH, filename), 'r') as file:
                current_time = None  # Reset for each file
                consider_log = True  # Flag to indicate if the file should be processed
                # Check every line in file to know where to stop reading file i.e. after current time crosses end time
                for line in file:
                    
                    # Check current time

                    match_CURRENT_TIME = pattern_CURRENT_TIME.search(line.strip())
                    
                    if match_CURRENT_TIME:
                        current_time = float(match_CURRENT_TIME.group(1))
                        # If current time is greater than or equal to END_TIME, skip the file
                        if current_time >= END_TIME: 
                            #print(filename)
                            #print(current_time)
                            #consider_log = False
                            break
                           # print(line)
                    if 'Neighbour Table after removing the addresses of the unreachable node' in line:
                        print(filename)
                        churn_count+=1
                        break

    # Calculate churn percentage
    if total_files > 0:
        churn_percent = (churn_count / total_files) * 100
    else:
        churn_percent = 0

    # Display the churn percentage
    print(f"CHURN PERCENTAGE: {churn_percent:.2f}%")

except Exception as e:
    print(f"An unexpected error occurred: {e}")


EIBP_C3.log
EIBP_C1.log
EIBP_A1.log
EIBP_D3.log
EIBP_A2.log
EIBP_D1.log
EIBP_D2.log
CHURN PERCENTAGE: 53.85%
